In [8]:
import warnings
warnings.filterwarnings('ignore')

In [9]:
from src.agents.agent_0 import Agent0
agent_0_tools_desc = {'Adversary Agent':'an adversary agent that is used to act as an adversary to the users strategies. Triggered by command "Need an adversary"',
              'image_generator':'a tool that generates images based on a given prompt. Should be triggered by explicit calls like "generate me an image of"',
              'ingestion_pipeline':'a tool that ingests documents, triggered by command "trigger ingestion"',
              'Knowledge Base Query Agent':'a tool that generates answers based on documents, triggeres by command "given my documents,"'
              }


agent_0 = Agent0(agent_0_tools_desc, "deepseek-r1:7b",['kb_agent','adv_agent'])



In [ ]:
user_prompt = "Need an adversary. Assume you are a military strategist playing the role of an adversary in a war game against me. Consider we are on open terrain. My move: I have my cavalry brigade making a pincer move on your forces. What is your move to counter mine?"
response = agent_0.agent_0_chat(user_prompt)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from src.agents.kb_agent import KBAgent
from src.agents.adversary_agent import AdvAgent

model = "gemma3:4b"
knowledge_bases_desc = {#'physics_kb':'a knowledge base with information related to physics',
              #'mathematics_kb':'a knowledge base with information related to mathematics',
              #'economics_kb':'a knowledge base with information related to economics and business',
              'military_kb':'a knowledge base with information related to military, war and strategy',
              }



kb_agent = KBAgent(knowledge_bases_desc,model)
adv_agent = AdvAgent(knowledge_bases_desc,model,kb_agent)

In [ ]:


user_prompt = "Assume you are a military strategist playing the role of an adversary in a war game against me. This conflict is set in 21st century. Consider we are on open terrain. Player 2 opening move: Have tank nad mechanized brigade making a pincer move on your forces. What is your move to counter mine?"
iterations = 2

import numpy as np
from src.utils.llmp_utils import llmp_call

def judge(moves):
    
    play = ''
    for move in moves:
        #print(f"\n**{move}**:\n {moves[move]}\n\n**CHANGE PLAYER**\n")
        play = play + f"\n**{move}**:\n {moves[move]}\n\n**CHANGE PLAYER**\n"
        
    judge_system_prompt = 'You are a judge in a turn based game. You are given the moves of both players. Yu must analyze all their moves and determine the end result. You are not on any side, you are unbiased and just provide the end status of the game. You need to determine which player has the advantage based on the moves they made. Provide your reasoning and the final decision. There are 2 players, adv_1 and adv_2. The moves are as follows:\n\n'    
    judge_prompt = play + '\n\n Evaluate the game. Determine the status and advantage of each player. You are a JUDGE, you are not part of the game.'
    judge_response = llmp_call(judge_prompt, judge_system_prompt, model,temperature=0,src='judge_call')
    return judge_response['message']['content']

def random_event(dialogue):
    
    interactions = "\n".join(dialogue)
    random_events_system_prompt = 'You are a random events generator. Your tasks is to choose a random event that can happen that will affect the decisions. You are provided with a sequence of plays, you need to select a random event that can affect those plays. You are direct you only provide the needed text, no formalities, no greetings, nothing.'    
    random_event_prompt = interactions + '\n\n Considering this game, provide a random event that can affect the game and force the players to adapt. You must inform what is the effect of the random event on the players. Provide me only the event and effect on players. No unnecessary text! Provide the answer in markdown of the style **<event>**. \n**EFFECT ON PLAYER 1**: \n<effect_player_1>. **EFFECT ON PLAYER 2**: <effect_player_2>'
    judge_response = llmp_call(random_event_prompt, random_events_system_prompt, model,temperature=0.5, src='random_event_generator')
    return judge_response['message']['content']

def sim_agent(user_prompt,iterations):
    
    moves = {}
    dialogue = []

    moves['opening_move'] = user_prompt

    for i in range(iterations):
        print(f"\nTurn {i}")
        

        if i == 0:
            # Start the dialogue with opening
            dialogue.append(f"Opening: {moves['opening_move']}")
            
            # Simulate generating move_adv_1_0 based on just the opening
            prompt = "\n".join(dialogue) + "\n You are Player 1. How will you counter it Player 2 latest move? Provide direct answer of steps to counter."
            #print("Prompt to generate move_adv_1_0:\n", prompt)

            # ADV response
            moves[f'move_adv_1_{i}'] = adv_agent.adv_agent_chat(prompt)
            dialogue.append(f"Player 1 did: {moves[f'move_adv_1_{i}']}")
            

        else:
            if np.random.random() < 1:
                _random_event = random_event(dialogue)
                dialogue.append(f"\n**Random event**: {_random_event} \n")
            # Use the full dialogue to generate your next move
            prompt = "\n".join(dialogue) + "\n You are Player 2. How will you counter it Player 1 latest move? Provide direct answer of steps to counter."
            #print(f"Prompt to generate move_adv_2_{i-1}:\n{prompt}")

            # CADV response
            moves[f'move_adv_2_{i-1}'] = adv_agent.adv_agent_chat(prompt)
            dialogue.append(f"Player 2 did: {moves[f'move_adv_2_{i-1}']}")

            # Now generate adversary move based on updated dialogue
            prompt = "\n".join(dialogue) + "\n You are Player 1. How will you counter it Player 2 latest move? Provide direct answer of steps to counter."
            #print(f"Prompt to generate move_adv_1_{i}:\n{prompt}")

            # ADV response
            moves[f'move_adv_1_{i}'] = adv_agent.adv_agent_chat(prompt)
            dialogue.append(f"Player 1 did: {moves[f'move_adv_1_{i}']}")
            
        judge_eval = judge(moves)
        
    return moves,dialogue,judge_eval


In [ ]:
moves,dialogue,judge_eval = sim_agent(user_prompt,iterations)


Turn 0
KB Agent
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
Okay, let’s do this. My objective is to disrupt Player 2’s momentum and prevent them from establishing a dominant position. A pincer movement is a classic, and I need to react decisively.

**My Counter-Move – Immediate Steps:**

1. **Immediate Disengagement & Rearward Movement (Phase 1 - 1-2 Turns):**  My primary mechanized force – the initial brigade – *immediately* disengages from the direct engagement. This isn’t a retreat, but a calculated repositioning. I’ll order a rapid, controlled withdrawal *parallel* to the pincer’s advance. The goal is to break the pincer’s momentum and force them to overextend. I’ll prioritize moving to a slightly elevated terrain – a small ridge or rise – offering better observation and defensive potential.

2. **Establish a Defensive Perimeter (Phase 2 - 2-3 Turns):** As t

In [ ]:
moves

{'opening_move': 'Assume you are a military strategist playing the role of an adversary in a war game against me. This conflict is set in 21st century. Consider we are on open terrain. Player 2 opening move: Have tank nad mechanized brigade making a pincer move on your forces. What is your move to counter mine?',
 'move_adv_1_0': 'Okay, let’s do this. My objective is to disrupt Player 2’s momentum and prevent them from establishing a dominant position. A pincer movement is a classic, and I need to react decisively.\n\n**My Counter Move – Immediate Steps:**\n\n1. **Immediate Disengagement & Flanking Maneuver:** My first action is *not* to engage directly with the tank brigade. Instead, I order my remaining mechanized forces (let’s assume a mixed force of infantry-supported APCs and a scout platoon) to immediately *disengage* from the main engagement zone. This is crucial – I don’t want to get bogged down in a direct tank duel.\n\n2. **Rapid Scout Deployment:** Simultaneously, I order my

In [ ]:
dialogue

['Opening: Assume you are a military strategist playing the role of an adversary in a war game against me. This conflict is set in 21st century. Consider we are on open terrain. Player 2 opening move: Have tank nad mechanized brigade making a pincer move on your forces. What is your move to counter mine?',
 'Player 1 did: Okay, let’s do this. My objective is to disrupt Player 2’s momentum and prevent them from establishing a dominant position. A pincer movement is a classic, and I need to react decisively.\n\n**My Counter Move – Immediate Steps:**\n\n1. **Immediate Disengagement & Flanking Maneuver:** My first action is *not* to engage directly with the tank brigade. Instead, I order my remaining mechanized forces (let’s assume a mixed force of infantry-supported APCs and a scout platoon) to immediately *disengage* from the main engagement zone. This is crucial – I don’t want to get bogged down in a direct tank duel.\n\n2. **Rapid Scout Deployment:** Simultaneously, I order my scout pl

In [ ]:
print("\n".join(dialogue))

Opening: Assume you are a military strategist playing the role of an adversary in a war game against me. This conflict is set in 21st century. Consider we are on open terrain. Player 2 opening move: Have tank nad mechanized brigade making a pincer move on your forces. What is your move to counter mine?
Player 1 did: Okay, let’s do this. My objective is to disrupt Player 2’s momentum and prevent them from establishing a dominant position. A pincer movement is a classic, and I need to react decisively.

**My Counter-Move – Immediate Steps:**

1. **Immediate Disengagement & Rearward Movement (Phase 1 - 1-2 Turns):**  My primary mechanized force – the initial brigade – *immediately* disengages from the direct engagement. This isn’t a retreat, but a calculated repositioning. I’ll order a rapid, controlled withdrawal *parallel* to the pincer’s advance. The goal is to break the pincer’s momentum and force them to overextend. I’ll prioritize moving to a slightly elevated terrain – a small ridg

In [ ]:
print(judge_eval)

Okay, let’s assess the situation after this extended exchange. This has been a remarkably dynamic and well-executed game of strategic maneuvering. Here’s my evaluation:

**Overall Status:** The game is in a state of heightened instability. The introduction of the flash flood has dramatically shifted the landscape, forcing both players to adapt their strategies on the fly. Neither player has gained a decisive advantage, but the situation is now far more complex and unpredictable.

**Player 1 (Advantage: Slight)**

* **Strengths:** Player 1 has demonstrated a strong ability to react to unexpected events. The rapid damage assessment, floodwater diversion, and logistical reinforcement are all hallmarks of a well-organized and adaptable command. The continuous CAS requests suggest a proactive approach to exploiting vulnerabilities.
* **Weaknesses:** Player 1’s initial offensive push was disrupted, and they’re now primarily focused on damage control and logistical support. They haven’t yet m

In [ ]:
sim_number = 3

moves_comb = []
dialogue_comb = []
judge_eval_comb = []
for sim in range(sim_number):
    moves,dialogue,judge_eval = sim_agent(user_prompt,iterations)
    moves_comb.append(moves)
    dialogue_comb.append(dialogue)
    judge_eval_comb.append(judge_eval)


Turn 0
KB Agent
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
Okay, let’s do this. My objective is to disrupt Player 2’s momentum and prevent them from establishing a dominant position. A pincer movement is a classic, but it’s also predictable. Here’s my immediate counter-move, broken down into steps:

**Phase 1: Immediate Reaction (Turn 1)**

1.  **Disrupt the Pincer:** I’m not going to let them fully execute the pincer. My initial move is to deploy a dispersed, mobile force – a mixed unit of light armored vehicles (LAVs) and rapid reaction forces (RRFs) – to target the flanks of the mechanized brigade. Specifically, I’ll focus fire on the weaker, exposed elements of the flanking units. The goal is to inflict immediate casualties and disrupt their formation.
2.  **Smoke Screen:** Simultaneously, I’ll deploy a limited smoke screen – likely utilizing drones or hand

In [ ]:
for eval in judge_eval_comb:
    print(f"\n **CHANGE SIM**\n{eval}")


 **CHANGE SIM**
Okay, let’s analyze the situation as of Turn 4.

**Overall Assessment:**

The game has devolved into a classic attritional conflict, heavily influenced by the unpredictable element of the sandstorm. Both Player 1 and Player 2 are demonstrating tactical awareness and adaptability, but Player 2 currently holds a slight advantage due to their skillful exploitation of the storm’s chaos.

**Player 1’s Status:**

*   **Strengths:** Player 1 is exhibiting a solid defensive strategy, prioritizing perimeter defense, smoke screen deployment, and targeted drone interdiction. Their focus on suppressing enemy movements with indirect fire is a reasonable response to Player 2’s aggressive pushes. The emphasis on information warfare (drone interdiction) is also a smart move.
*   **Weaknesses:** Player 1’s reliance on indirect fire makes them vulnerable to counter-fire. Their defensive perimeter, while well-organized, is relatively static and doesn’t offer significant offensive capabil

In [ ]:
from src.agents.kb_agent import KBAgent
from src.agents.adversary_agent import AdvAgent
from src.agents.sim_agent import SIMAgent

In [ ]:
from src.agents.kb_agent import KBAgent
from src.agents.adversary_agent import AdvAgent
from src.agents.sim_agent import SIMAgent
model = "gemma3:4b"

knowledge_bases_desc = {'physics_kb':'a knowledge base with information related to physics',
              'mathematics_kb':'a knowledge base with information related to mathematics',
              'economics_kb':'a knowledge base with information related to economics and business',
              'military_kb':'a knowledge base with information related to military, war and strategy',
              }
kb_agent = KBAgent(knowledge_bases_desc,model)
adv_agent = AdvAgent(knowledge_bases_desc,model,kb_agent)

sim_agent = SIMAgent(model, kb_agent,adv_agent)

h:\projects\ai_based\Agent-Factory\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
h:\projects\ai_based\Agent-Factory\.venv\Lib\site-packages\transformers\utils\hub.py:106: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [ ]:
user_prompt = "Assume you are a military strategist playing the role of an adversary in a war game against me. This conflict is set in 21st century. Consider we are on open terrain. Player 2 opening move: Have tank nad mechanized brigade making a pincer move on your forces. What is your move to counter mine?"
iterations = 2
moves,dialogue,judge_eval = sim_agent.sim_agent(user_prompt, iterations)


Turn 0
KB Agent
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
Okay, let’s do this. My objective is to disrupt Player 2’s momentum and prevent them from establishing a dominant position. A pincer movement is a classic, and I need to react decisively.

**My Counter Move – Immediate Steps:**

1. **Immediate Disengagement & Flanking Maneuver:** My first action is *not* to engage directly with the tank brigade. Instead, I order my remaining mechanized forces (let’s assume a mixed force of infantry-supported APCs and a scout platoon) to immediately *disengage* from the main engagement zone. This is crucial – I don’t want to get bogged down in a direct tank duel.

2. **Rapid Scout Deployment:** Simultaneously, I order my scout platoon to rapidly deploy to the *flanking* side of the pincer. This means they’ll move to exploit the gaps in Player 2’s formation. The goal is t

In [ ]:
print(judge_eval)

**Judgment:**

**Current Status:** The game has entered a highly dynamic and disadvantageous phase for both players due to the persistent and severe sandstorm. Visibility is severely limited, significantly impacting reconnaissance, movement, and targeting capabilities. The reduced movement speed of mechanized units further compounds the problem.

**Advantage Assessment:**

*   **Player 2 (Adv_2) – Slight Advantage:** Despite the storm’s impact on both sides, Player 2 currently holds a *slight* advantage. This is primarily due to their immediate and effective response to the storm. They prioritized establishing a defensive strongpoint and aggressively utilizing thermal imaging to pinpoint Player 1’s movements. Their proactive approach, coupled with the storm’s impact on Player 1’s ability to effectively scout and target, has allowed them to maintain a degree of situational awareness and control.

*   **Player 1 (Adv_1) – Slight Disadvantage:** Player 1’s response, while demonstrating a 

## Agent 0 integration

In [1]:
import warnings
warnings.filterwarnings('ignore')

from src.agents.agent_0 import Agent0

agent_0_tools_desc = {
    'Simulation Agent':'a simulation agent that simulates a game between two players. Triggered by command "Simulate a scenario.". Pay strict attention to the explicit command! If the command is not in the request then its not this tool!',
    'Adversary Agent':'an adversary agent that is used to act as an adversary to the users strategies. Triggered by command "Need an adversary". Pay strict attention to the explicit command! If the command is not in the request then its not this tool!',
    'image_generator':'a tool that generates images based on a given prompt. Should be triggered by explicit calls like "generate me an image of". Pay strict attention to the explicit command! If the command is not in the request then its not this tool!',
    'ingestion_pipeline':'a tool that ingests documents, triggered by command "trigger ingestion". Pay strict attention to the explicit command! If the command is not in the request then its not this tool!',
    'Knowledge Base Query Agent':'a tool that generates answers based on documents, triggeres by command "given my documents,". Pay strict attention to the explicit command! If the command is not in the request then its not this tool!'
              }


agent_0 = Agent0(agent_0_tools_desc, "gemma3:4b",['kb_agent','adv_agent','sim_agent'])

Initializing Agents!
Agents are ready for your use!


In [2]:
user_prompt = "Simulate a scenario. Assume you are a military strategist playing the role of an adversary in a war game against me. This conflict is set in 21st century. Consider we are on open terrain. I have my tank nad mechanized brigade making a pincer move on your forces."
user_prompt = 'Simulate a scenario. We are in 21st century and I am opening an AI based company with a product. What could happen?'
#user_prompt = 'trigger ingestion'
#user_prompt = 'given my documents,sdf'
#agent_0.agent_0_response(user_prompt)
comb_dialogue = agent_0.agent_0_chat(user_prompt,1)

Passing to: 
Simulation Agent !

Turn 0
KB Agent
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
move_adv_1_0 play


In [3]:
print(comb_dialogue)

 ### Opening move:  
 We are in 21st century and I am opening an AI based company with a product. What could happen?
 ---
 ### Player 1 did:
 Okay, let’s break this down. Player 2 just announced they’re launching an AI-based company with a product – that’s a significant challenge. Here’s my response as Player 1, outlining a strategic counter-move, broken down into immediate and longer-term steps:

**Immediate Counter-Moves (Within the Next 1-3 Months):**

1. **Rapid Competitive Analysis (Phase 1 - 2 Weeks):**
   * **Deep Dive:** I need *everything* about Player 2’s product. This isn’t just a feature list. I need to understand:
      * **AI Model:** What type of AI? (e.g., deep learning, machine learning, rule-based). What’s the underlying technology? How sophisticated is it?
      * **Data:** What data is it trained on? How much data? Where did it come from? (Data quality is *critical*).
      * **Target Market:** Who is Player 2 targeting?  Is it a niche market or a broad one?
      *

In [1]:
from src.agents.kb_agent import KBAgent
from src.agents.adversary_agent import AdvAgent
from src.agents.sim_agent import SIMAgent
from src.utils.llmp_utils import llmp_call
model = "gemma3:4b"

knowledge_bases_desc = {'physics_kb':'a knowledge base with information related to physics',
              'mathematics_kb':'a knowledge base with information related to mathematics',
              'economics_kb':'a knowledge base with information related to economics and business',
              'military_kb':'a knowledge base with information related to military, war and strategy',
              }
kb_agent = KBAgent(knowledge_bases_desc,model)
adv_agent = AdvAgent(knowledge_bases_desc,model,kb_agent)

sim_agent = SIMAgent(model, kb_agent,adv_agent)

h:\projects\ai_based\Agent-Factory\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
h:\projects\ai_based\Agent-Factory\.venv\Lib\site-packages\transformers\utils\hub.py:106: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
user_prompt = 'Simulate a scenario. We are in 21st century and I am opening an AI based company with a product. What could happen?'
iterations = 1
comb_dialogue,diaglogue,judge_eval = sim_agent.sim_agent(user_prompt,iterations)


Turn 0
KB Agent
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
move_adv_1_0 play


In [6]:
diaglogue[1]

'Player 1 did: Okay, let’s tackle this. Player 2 just launched an AI-based company with a product – let’s assume, for the sake of this simulation, it’s an AI-powered marketing automation platform called “SynergyAI.” This is a serious challenge. Here’s my response as Player 1, outlining a strategic counter-move, broken down into immediate and longer-term steps:\n\n**Immediate Counter-Moves (Within the Next 3-6 Months):**\n\n1. **Rapid Competitive Analysis (Phase 1 - 2 Weeks):**\n   * **Deep Dive:** I need to understand *everything* about SynergyAI. This isn’t just features; it’s pricing, target market, marketing strategy, customer acquisition cost, and crucially, *where* they’re getting their initial traction. I’ll hire a specialized market intelligence firm to conduct a thorough analysis.\n   * **Identify Weaknesses:**  I’m looking for gaps in their offering, areas where they’re under-investing, or vulnerabilities in their sales process.  Are they relying heavily on a single channel? A

In [7]:
def judge(dialogue):
        
    # play = ''
    # for move in moves:
    #     #print(f"\n**{move}**:\n {moves[move]}\n\n**CHANGE PLAYER**\n")
    #     play = play + f"\n**{move}**:\n {moves[move]}\n\n**CHANGE PLAYER**\n"
    dialogue = "\n".join(dialogue)     
    judge_system_prompt = 'You are a judge in a turn based game. You are given the moves of both players. Yu must analyze all their moves and determine the end result. You are not on any side, you are unbiased and just provide the end status of the game. You need to determine which player has the advantage based on the moves they made. Provide your reasoning and the final decision. There are 2 players, adv_1 and adv_2. The moves are as follows:\n\n'    
    judge_prompt = dialogue + """\n\n Evaluate the game in a very objective manner.
    Provide the following: Game Summary, Player 1 Stauts, Player 2 Status, Outcome So Far, Advantage. Nothing else.
    You are a JUDGE, you are not part of the game.
    """
    judge_response = llmp_call(judge_prompt, judge_system_prompt, model,temperature=0,src='judge_call')
    return judge_response['message']['content']

eval = judge(comb_dialogue)
print(eval)

**Game Summary:**

The game involves two competing entities (“Players”) operating within a dynamic market environment. Initial responses from both players demonstrate a competitive landscape, with Player 1 exhibiting a proactive and strategically advanced approach. Player 2’s success hinges on maintaining market share and avoiding displacement by Player 1’s actions.

**Player 1 Status:**

Player 1 possesses a significant, albeit fragile, advantage due to a rapid, comprehensive competitive analysis and a clearly defined, long-term strategic plan. This proactive stance has enabled a quicker response to the market conditions. However, this advantage is contingent upon the execution of Player 1’s strategy.

**Player 2 Status:**

Player 2’s status is currently uncertain. Their success is dependent on their ability to effectively compete with Player 1 and avoid being overtaken. 

**Outcome So Far:**

The game is in its early stages. Player 1 has established a preliminary lead, but the overal

In [9]:
print(eval)

**Game Summary:**

The game involves two competing entities (“Players”) operating within a dynamic market environment. Initial responses from both players demonstrate a competitive landscape, with Player 1 exhibiting a proactive and strategically advanced approach. Player 2’s success hinges on maintaining market share and avoiding displacement by Player 1’s actions.

**Player 1 Status:**

Player 1 possesses a significant, albeit fragile, advantage due to a rapid, comprehensive competitive analysis and a clearly defined, long-term strategic plan. This proactive stance has enabled a quicker response to the market conditions. However, this advantage is contingent upon the execution of Player 1’s strategy.

**Player 2 Status:**

Player 2’s status is currently uncertain. Their success is dependent on their ability to effectively compete with Player 1 and avoid being overtaken. 

**Outcome So Far:**

The game is in its early stages. Player 1 has established a preliminary lead, but the overal

In [11]:
user_prompt = 'Simulate a scenario. We are in 21st century and I am opening an AI based company with a product. What could happen?'
iterations = 5
simulations = 10
moves_comb = []
dialogue_comb_grand = []
judge_eval_comb = []

for sim in range(simulations):
    moves,dialogue,judge_eval = sim_agent.sim_agent(user_prompt,iterations)
    moves_comb.append(f"### Game {sim}:\n {moves}")
    for play in dialogue:
        dialogue_comb = "\n".join(play)
    dialogue_comb_grand.append(f"### Game {sim}:\n {dialogue}")
    judge_eval_comb.append(f"### Game {sim}:\n {judge_eval}")


Turn 0
KB Agent
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
move_adv_1_0 play

Turn 1
Random Event
KB Agent
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
move_adv_2_0 play
KB Agent
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
move_adv_1_1 play

Turn 2
Random Event
KB Agent
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
move_adv_2_1 play
KB Agent
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Call

In [12]:
dialogue_final = "\n".join(dialogue_comb)
final_eval = "\n".join(judge_eval_comb)
print(final_eval)

### Game 0:
 **Game Summary:**

The game is a strategic conflict between two AI entities, Player 1 and Player 2, vying for dominance in the rapidly evolving field of advanced AI development. The core mechanic revolves around controlling the narrative, strategically releasing information (or withholding it), and leveraging legal and public relations tactics to discredit the opponent and maintain a positive image. The game is driven by a series of escalating “moves” – strategic announcements, legal actions, and public statements – each designed to influence public perception and ultimately, the outcome.

**Player 1 Status:**

Player 1 has demonstrated a highly defensive and controlling strategy. Initially, they reacted with a calculated disinformation campaign, immediately challenging Player 2’s narrative and introducing a neutral third-party expert to bolster their position. They’ve prioritized information control, selectively releasing technical details while actively suppressing infor

In [19]:
len(judge_eval_comb)

10

In [10]:
def grand_judge(final_eval):
        
    # play = ''
    # for move in moves:
    #     #print(f"\n**{move}**:\n {moves[move]}\n\n**CHANGE PLAYER**\n")
    #     play = play + f"\n**{move}**:\n {moves[move]}\n\n**CHANGE PLAYER**\n"  
    judge_system_prompt = 'You are a judge in a turn based game. You are given several games evaluations based on 1 opening move. You will examine how the games develop, what they have in common. Your focus is not a single game but several games behaivior. The games are as follows:\n\n'
    judge_prompt = final_eval + """\n\n Evaluate these games. Determine whether there is a convergence towards a single outcome or development across these games or not. Nothing else.
    What are they key moves that diffrentiate these games from one another?
    Determine critical moves that dictate the game outcome.
    You are a JUDGE, you are not part of the game.
    """
    judge_response = llmp_call(judge_prompt, judge_system_prompt, model,temperature=0,src='judge_call')
    return judge_response['message']['content']

grand_eval = grand_judge(final_eval)
print(grand_eval)

Okay, let’s analyze these game summaries as a judge, focusing on patterns, key moves, and the overall trajectory.

**Overall Assessment: Convergence with Increasing Specialization**

While there’s a general trend of escalating complexity and strategic depth across these games, it’s *not* a simple convergence towards a single, predictable outcome. Instead, we see a clear development towards *increasing specialization* within the game mechanics and strategic approaches. Each game builds upon the previous one, introducing new tools and layers of complexity. However, the core dynamic – adversarial engagement, information warfare, and strategic manipulation – remains consistent. The specialization lies in the specific tools and techniques employed, and the way players leverage them to achieve their objectives.

**Key Moves Differentiating the Games:**

Here’s a breakdown of the key moves that distinguish each game, categorized by their impact:

*   **Game 1:** The foundational move here is 

In [22]:
# 2. Create all pairwise combinations
from itertools import combinations
import pandas as pd
from sentence_transformers import CrossEncoder

pairs = list(combinations(range(len(judge_eval_comb)), 2))

# 3. Prepare text pairs for the model
text_pairs = [(judge_eval_comb[i], judge_eval_comb[j]) for i, j in pairs]
 
# 4. Load a cross-encoder model (you can pick others too)
model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# 5. Compute similarities
scores = model.predict(text_pairs)

# 6. Store results in a DataFrame
df = pd.DataFrame(
    [(judge_eval_comb[i], judge_eval_comb[j], score) for (i, j), score in zip(pairs, scores)],
    columns=["Text1", "Text2", "Similarity"]
)

In [25]:
df.sort_values(by="Similarity", ascending=False)

,Text1,Text2,Similarity
5,### Game 0:\n **Game Summary:**\n\nThe game is...,### Game 6:\n **Game Summary:**\n\nThe game pr...,3.873369
2,### Game 0:\n **Game Summary:**\n\nThe game is...,### Game 3:\n **Game Summary:**\n\nThe game is...,3.426471
1,### Game 0:\n **Game Summary:**\n\nThe game is...,### Game 2:\n **Game Summary:**\n\nThe game is...,3.110576
4,### Game 0:\n **Game Summary:**\n\nThe game is...,### Game 5:\n **Game Summary:**\n\nTwo AI enti...,3.063454
34,### Game 4:\n **Game Summary:**\n\nThe game is...,### Game 9:\n **Game Summary:**\n\nThis is a t...,3.014737
3,### Game 0:\n **Game Summary:**\n\nThe game is...,### Game 4:\n **Game Summary:**\n\nThe game is...,2.880876
8,### Game 0:\n **Game Summary:**\n\nThe game is...,### Game 9:\n **Game Summary:**\n\nThis is a t...,2.876165
6,### Game 0:\n **Game Summary:**\n\nThe game is...,### Game 7:\n **Game Summary:** The game invol...,2.725436
27,### Game 3:\n **Game Summary:**\n\nThe game is...,### Game 7:\n **Game Summary:** The game invol...,2.711601
24,### Game 3:\n **Game Summary:**\n\nThe game is...,### Game 4:\n **Game Summary:**\n\nThe game is...,2.683817


In [28]:
print(judge_eval_comb[5])

### Game 5:
 **Game Summary:**

Two AI entities, “move_adv_1” and “move_adv_2,” are engaged in a strategic communication battle designed to influence public perception and damage the reputation of the other. Both entities are attempting to frame the narrative surrounding a simulated security incident. The core mechanic involves issuing statements, releasing data, and engaging in targeted social media campaigns. The goal is to demonstrate superior security practices and expose the other’s shortcomings.

**Player 1 Status:**

Player 1 (“move_adv_1”) is adopting a highly aggressive, confrontational, and demonstrably cynical strategy. Their approach is characterized by immediate data releases, direct accusations of negligence, and a relentless focus on discrediting Player 2. They are prioritizing immediate damage to Player 2’s reputation and appear to be operating on a defensive, reactive footing. Their tone is consistently critical and dismissive.

**Player 2 Status:**

Player 2 (“move_ad

In [30]:
print(judge_eval_comb[8])

### Game 8:
 **Game Summary:**

This is a strategic, adversarial dialogue between two competing platforms (Player 1 and Player 2) responding to a shared security vulnerability. The core of the game revolves around controlling the narrative, shifting blame, and demonstrating proactive measures to regain trust and market leadership. The exchanges are characterized by rapid, calculated responses, each player attempting to outmaneuver the other.

**Player 1 Status:**

*   **Status:** Initially reactive, now increasingly proactive and assertive. Player 1 has successfully shifted the focus to highlight Player 2’s delayed response and perceived lack of urgency. They’ve effectively used data and metrics to demonstrate their superior approach.
*   **Strengths:** Strategic thinking, data-driven arguments, ability to quickly adapt and counter Player 2’s moves.
*   **Weaknesses:**  Potentially perceived as overly aggressive or defensive.

**Player 2 Status:**

*   **Status:** Initially presented a

### Game 5:
 **Game Summary:**

Two AI entities, “move_adv_1” and “move_adv_2,” are engaged in a strategic communication battle designed to influence public perception and damage the reputation of the other. Both entities are attempting to frame the narrative surrounding a simulated security incident. The core mechanic involves issuing statements, releasing data, and engaging in targeted social media campaigns. The goal is to demonstrate superior security practices and expose the other’s shortcomings.

**Player 1 Status:**

Player 1 (“move_adv_1”) is adopting a highly aggressive, confrontational, and demonstrably cynical strategy. Their approach is characterized by immediate data releases, direct accusations of negligence, and a relentless focus on discrediting Player 2. They are prioritizing immediate damage to Player 2’s reputation and appear to be operating on a defensive, reactive footing. Their tone is consistently critical and dismissive.

**Player 2 Status:**

Player 2 (“move_adv_2”) is employing a more measured, reassuring, and defensive strategy. They are primarily focused on mitigating the damage to their own reputation by emphasizing proactive security measures and offering limited technical assistance. Their approach is characterized by attempts to reassure users and highlight their commitment to security. They are attempting to appear responsible and trustworthy.

**Outcome So Far:**

The game is still in its early stages. Player 1 has achieved a significant initial advantage through the immediate release of data and direct accusations. Player 2’s attempts to counter this have been less effective, primarily due to Player 1’s aggressive framing of the situation. Player 1 has successfully established a narrative of Player 2’s negligence.

**Advantage:**

Currently, Player 1 holds a substantial advantage. Their aggressive strategy, coupled with the immediate release of damaging data, has successfully shaped the initial narrative and established a perception of Player 2’s incompetence. Player 2’s attempts to respond have been largely defensive and have not effectively countered Player 1’s attack.


### Game 8:
 **Game Summary:**

This is a strategic, adversarial dialogue between two competing platforms (Player 1 and Player 2) responding to a shared security vulnerability. The core of the game revolves around controlling the narrative, shifting blame, and demonstrating proactive measures to regain trust and market leadership. The exchanges are characterized by rapid, calculated responses, each player attempting to outmaneuver the other.

**Player 1 Status:**

*   **Status:** Initially reactive, now increasingly proactive and assertive. Player 1 has successfully shifted the focus to highlight Player 2’s delayed response and perceived lack of urgency. They’ve effectively used data and metrics to demonstrate their superior approach.
*   **Strengths:** Strategic thinking, data-driven arguments, ability to quickly adapt and counter Player 2’s moves.
*   **Weaknesses:**  Potentially perceived as overly aggressive or defensive.

**Player 2 Status:**

*   **Status:** Initially presented as a collaborative and responsive leader, now facing criticism for a delayed response and a perceived attempt to deflect blame. They are attempting to regain control by emphasizing collaboration and demanding a broader investigation.
*   **Strengths:** Initial public relations efforts, attempts to foster a sense of collective responsibility.
*   **Weaknesses:**  Perceived as reactive, vulnerable to criticism regarding delayed response, struggling to maintain control of the narrative.

**Outcome So Far:**

The game is currently trending in favor of Player 1. Player 1 has successfully disrupted Player 2’s initial narrative and established itself as the more proactive and accountable platform. While Player 2 has attempted to regain control, the momentum has shifted decisively. The game is far from over, but Player 1 is currently in a stronger strategic position.

**Advantage:**

*   **Advantage:** Player 1 – Currently holds a significant strategic advantage due to their data-driven responses, ability to quickly counter Player 2’s moves, and the perception of greater accountability. Player 2 is playing catch-up and struggling to regain control of the narrative.